# Assignment 1 — Question 4
**Student ID:** 26K-7618

Binary vectors over the field $\mathbb{F}_2 = \{0, 1\}$, where addition is done modulo 2:

$$0 + 0 = 0,\quad 0 + 1 = 1,\quad 1 + 0 = 1,\quad 1 + 1 = 0$$

## Setup

In [5]:
from itertools import product

import numpy as np

# the three given vectors
x = np.array([1, 0, 1, 1, 0])
y = np.array([0, 1, 1, 0, 1])
z = np.array([1, 1, 0, 1, 1])

vecs = [x, y, z]
names = ["x", "y", "z"]
size = len(x)


def add(p, q):
    """Addition in F2 (mod 2)."""
    return (p + q) % 2


print("x =", x)
print("y =", y)
print("z =", z)

x = [1 0 1 1 0]
y = [0 1 1 0 1]
z = [1 1 0 1 1]


## (a) Compute $x + y$ over $\mathbb{F}_2$

In [6]:
x_plus_y = add(x, y)
print("x + y =", x_plus_y)

# position by position
for i in range(size):
    print(f"  {x[i]} + {y[i]} = {x[i] + y[i]} mod 2 = {x_plus_y[i]}")

x + y = [1 1 0 1 1]
  1 + 0 = 1 mod 2 = 1
  0 + 1 = 1 mod 2 = 1
  1 + 1 = 2 mod 2 = 0
  1 + 0 = 1 mod 2 = 1
  0 + 1 = 1 mod 2 = 1


## (b) Are $x, y, z$ linearly independent over $\mathbb{F}_2$?

The only scalars in $\mathbb{F}_2$ are 0 and 1, so a linear combination just means picking some of the vectors and adding them. We find the rank using Gaussian elimination mod 2.

In [8]:
def rank(mat):
    """Rank of a binary matrix over F2."""
    m = mat.copy() % 2
    rows, cols = m.shape
    r = 0

    for col in range(cols):
        # find a row at or below r with a 1 in this column
        pivot = -1
        for i in range(r, rows):
            if m[i, col] == 1:
                pivot = i
                break
        if pivot == -1:
            continue

        m[[r, pivot]] = m[[pivot, r]]          # swap it up
        for i in range(rows):                  # clear the rest of the column
            if i != r and m[i, col] == 1:
                m[i] = add(m[i], m[r])
        r += 1

    return r, m


mat = np.array(vecs)
r, reduced = rank(mat)

print("matrix [x; y; z]:")
print(mat)
print("\nafter elimination mod 2:")
print(reduced)
print("\nrank =", r)

if r < 3:
    print("=> x, y, z are LINEARLY DEPENDENT over F2")
    print("   x + y + z =", add(add(x, y), z), "(zero vector)")
    print("   so z = x + y")
else:
    print("=> x, y, z are linearly independent over F2")

matrix [x; y; z]:
[[1 0 1 1 0]
 [0 1 1 0 1]
 [1 1 0 1 1]]

after elimination mod 2:
[[1 0 1 1 0]
 [0 1 1 0 1]
 [0 0 0 0 0]]

rank = 2
=> x, y, z are LINEARLY DEPENDENT over F2
   x + y + z = [0 0 0 0 0] (zero vector)
   so z = x + y


## (c) All vectors in $\text{span}(x, y, z)$

We try all 8 coefficient choices $(c_1, c_2, c_3)$ and compute $c_1 x + c_2 y + c_3 z$. Since $z = x + y$, several choices give the same vector.

In [9]:
span = set()

print("coefficients   combination          result")
for coeffs in product([0, 1], repeat=3):
    total = np.zeros(size, dtype=int)
    for c, v in zip(coeffs, vecs):
        if c == 1:
            total = add(total, v)
    span.add(tuple(int(b) for b in total))

    label = " + ".join(n for c, n in zip(coeffs, names) if c) or "0"
    print(f"{str(coeffs):<15}{label:<21}{total}")

print(f"\ndistinct vectors in the span: {len(span)}")
for v in sorted(span):
    print("  ", v)

coefficients   combination          result
(0, 0, 0)      0                    [0 0 0 0 0]
(0, 0, 1)      z                    [1 1 0 1 1]
(0, 1, 0)      y                    [0 1 1 0 1]
(0, 1, 1)      y + z                [1 0 1 1 0]
(1, 0, 0)      x                    [1 0 1 1 0]
(1, 0, 1)      x + z                [0 1 1 0 1]
(1, 1, 0)      x + y                [1 1 0 1 1]
(1, 1, 1)      x + y + z            [0 0 0 0 0]

distinct vectors in the span: 4
   (0, 0, 0, 0, 0)
   (0, 1, 1, 0, 1)
   (1, 0, 1, 1, 0)
   (1, 1, 0, 1, 1)


## (d) Dimension of the subspace

The dimension of a span equals the rank. A subspace of dimension $k$ over $\mathbb{F}_2$ contains exactly $2^k$ vectors, which we can use as a check.

In [10]:
dim = r
print("dimension =", dim)
print(f"check: 2^{dim} = {2 ** dim}, and we found {len(span)} vectors")
print(f"\nbasis: x = {x}, y = {y}")
print("z is redundant because z = x + y")

dimension = 2
check: 2^2 = 4, and we found 4 vectors

basis: x = [1 0 1 1 0], y = [0 1 1 0 1]
z is redundant because z = x + y


## (e) Relevance to error detection and correction

Binary vectors over $\mathbb{F}_2$ are the basis of linear block codes used in data transmission. The set of valid codewords is a subspace, exactly like the span found in part (c): a $k$-dimensional subspace of $\mathbb{F}_2^n$ gives an $(n,k)$ code that encodes $k$ information bits into $n$ transmitted bits. Linear independence decides how much information the code actually carries — here $z = x + y$, so $z$ is dependent and adds nothing new, leaving a dimension of 2 instead of 3.

The gap between the $2^k$ valid codewords and all $2^n$ possible bit patterns is the redundancy that makes error handling possible. If noise flips bits during transmission, the received vector usually falls outside the subspace, so the receiver detects the error by checking whether it still satisfies the parity equations. With enough redundancy (a large enough minimum Hamming distance) the receiver can also pick the nearest valid codeword and correct the error, which is exactly how Hamming codes and other linear error-correcting codes work.